# 08 · Near-duplicate check and four-class parameter counts (no training)

Two CPU-only checks that the byte-level MD5 scan of the paper cannot do.

1. **Perceptual-hash near-duplicates.** MD5 finds only byte-identical files. A 64-bit DCT perceptual hash (pHash) also finds re-encoded, resized or lightly edited copies. The notebook reports, for several Hamming-distance thresholds,
   * BRISC2025 **test** images that have a near-duplicate in the **training** pool (internal leakage),
   * near-duplicate pairs **inside** the training pool,
   * images of each **external** set that have a near-duplicate in BRISC2025, and how many of those are *not* byte-identical (the part MD5 missed).
2. **Parameter counts** of MobileNetV4-Small and RepViT-M1 (timm, the constructors used in notebooks 02-03) with a four-class head and with the library 1000-class head.

pHash detects visual similarity, not patient identity. A small distance means "very similar image", so the counts are a screen for leakage, not proof of it. Outputs go to `/kaggle/working`.

**Attach:** BRISC2025 and the external Mendeley sets (any that are missing are skipped).

**Revision.** Added: saved nearest-training-image pairs for every test image (with figures), calibration of the pHash threshold (true-copy detection vs cross-class chance matches), and group assignment. The accuracy of matched vs unmatched test images is reported in notebook 01 and in section 8 of notebook 07.

In [ ]:
!pip install -q timm
import os, hashlib, math, itertools
from pathlib import Path
import numpy as np, pandas as pd, cv2
OUT_DIR = Path("/kaggle/working"); OUT_DIR.mkdir(exist_ok=True, parents=True)
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

In [ ]:
BRISC_ROOT = Path("/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task")
BRISC_TRAIN_ROOT, BRISC_TEST_ROOT = BRISC_ROOT / "train", BRISC_ROOT / "test"
assert BRISC_TRAIN_ROOT.exists() and BRISC_TEST_ROOT.exists(), "Attach BRISC2025 (classification_task) to this notebook."

# Same external roots as notebook 07. Missing ones are skipped.
_ds3 = Path("/kaggle/input/datasets/neyamul/bt-mendeley-3")
_ds3_sub = [d for d in _ds3.iterdir() if d.is_dir()] if _ds3.exists() else []
EXTERNAL = {
    "mendeley3": (_ds3_sub[0] / "Test") if len(_ds3_sub) == 1 else None,
    "mendeley5": Path("/kaggle/input/datasets/neyamul/bt-mendeley-5/Brain Tumor data/Testing"),
    "mendeley_epic_cscr": Path("/kaggle/input/datasets/neyamul/mendeley-bt-data/Epic and CSCR hospital Dataset/Test"),
    "mendeley2": Path("/kaggle/input/datasets/neyamul/mendeley-2/test"),
}

def list_images(root):
    return sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in IMG_EXT)

In [ ]:
def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def phash64(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    low = cv2.dct(img)[:8, :8].flatten()
    med = np.median(low[1:])                      # ignore the DC term
    bits = (low > med).astype(np.uint64)
    return np.uint64(sum(int(b) << i for i, b in enumerate(bits)))

def describe(paths, name):
    rows = []
    for p in paths:
        h = phash64(p)
        if h is not None:
            rows.append((str(p), md5_of(p), h))
    df = pd.DataFrame(rows, columns=["path", "md5", "phash"])
    df["set"] = name
    print(f"{name}: {len(df)} images hashed (of {len(paths)})")
    return df

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def min_hamming(Q, R, exclude_self=False, chunk=256):
    """For every hash in Q the smallest Hamming distance to any hash in R."""
    Q = np.asarray(Q, dtype=np.uint64); R = np.asarray(R, dtype=np.uint64)
    out = np.empty(len(Q), dtype=np.int16)
    for s in range(0, len(Q), chunk):
        x = (Q[s:s + chunk, None] ^ R[None, :]).view(np.uint8).reshape(-1, len(R), 8)
        d = POP[x].sum(-1).astype(np.int16)
        if exclude_self:
            for k in range(d.shape[0]):
                d[k, s + k] = 99
        out[s:s + chunk] = d.min(1)
    return out

In [ ]:
train_df = describe(list_images(BRISC_TRAIN_ROOT), "brisc_train_pool")
test_df  = describe(list_images(BRISC_TEST_ROOT),  "brisc_test")
THRESH = [0, 2, 4, 6, 8, 10]

# (a) test images with a near-duplicate in the training pool
d_test = min_hamming(test_df["phash"].values, train_df["phash"].values)
test_df["min_dist_to_train"] = d_test
test_df["md5_in_train"] = test_df["md5"].isin(set(train_df["md5"]))
# (b) near-duplicates inside the training pool (each image against all others)
d_train = min_hamming(train_df["phash"].values, train_df["phash"].values, exclude_self=True)
train_df["min_dist_to_other_train"] = d_train

rows = []
for t in THRESH:
    rows.append({"threshold (Hamming <=)": t,
                 "test images with a train near-duplicate": int((d_test <= t).sum()),
                 "% of test": round(100 * (d_test <= t).mean(), 2),
                 "of which not byte-identical": int(((d_test <= t) & ~test_df["md5_in_train"]).sum()),
                 "train images with another train near-duplicate": int((d_train <= t).sum()),
                 "% of train": round(100 * (d_train <= t).mean(), 2)})
internal = pd.DataFrame(rows); print(internal.to_string(index=False))
internal.to_csv(OUT_DIR / "phash_internal_summary.csv", index=False)
test_df.drop(columns=["phash"]).to_csv(OUT_DIR / "phash_test_vs_train.csv", index=False)

In [ ]:
ext_rows = []
brisc_all = pd.concat([train_df, test_df], ignore_index=True)
brisc_md5 = set(brisc_all["md5"])
for name, root in EXTERNAL.items():
    if root is None or not Path(root).exists():
        print(f"[skip] {name}: not attached"); continue
    e = describe(list_images(root), name)
    e["min_dist_to_brisc"] = min_hamming(e["phash"].values, brisc_all["phash"].values)
    e["md5_in_brisc"] = e["md5"].isin(brisc_md5)
    e.drop(columns=["phash"]).to_csv(OUT_DIR / f"phash_{name}_vs_brisc.csv", index=False)
    for t in THRESH:
        near = e["min_dist_to_brisc"] <= t
        ext_rows.append({"external set": name, "n": len(e), "threshold (Hamming <=)": t,
                         "byte-identical (MD5)": int(e["md5_in_brisc"].sum()),
                         "near-duplicate (pHash)": int(near.sum()),
                         "near-duplicate but not byte-identical": int((near & ~e["md5_in_brisc"]).sum()),
                         "no near-duplicate": int((~near).sum())})
external = pd.DataFrame(ext_rows)
print(external.to_string(index=False))
external.to_csv(OUT_DIR / "phash_external_summary.csv", index=False)

## Matched pairs (inspect them)
The nearest training image of every test image is saved, with one figure per distance bin (`phash_pairs_d*.png`) and a table of how often the two images carry the same class label. A bin in which almost every pair has the same label and looks like the same slice contains true duplicates.

In [ ]:
# ---- Inspect matched pairs: save the nearest training image of each test image, by distance bin ----
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def hash_arrays(df):
    return np.array([int(x) for x in df["phash"].values], dtype=np.uint64)

def nearest_with_index(Q, R, chunk=256):
    Q = np.asarray(Q, dtype=np.uint64); R = np.asarray(R, dtype=np.uint64)
    dist = np.empty(len(Q), dtype=np.int16); idx = np.empty(len(Q), dtype=np.int64)
    for s in range(0, len(Q), chunk):
        x = (Q[s:s + chunk, None] ^ R[None, :]).view(np.uint8).reshape(-1, len(R), 8)
        d = POP[x].sum(-1).astype(np.int16)
        j = d.argmin(1); dist[s:s + len(d)] = d[np.arange(len(d)), j]; idx[s:s + len(d)] = j
    return dist, idx

train_df = train_df.reset_index(drop=True); test_df = test_df.reset_index(drop=True)
nn_dist, nn_idx = nearest_with_index(test_df["phash"].values, train_df["phash"].values)
cls_of = lambda p: Path(p).parent.name
pairs = pd.DataFrame({"test_path": test_df["path"], "train_path": train_df["path"].values[nn_idx], "distance": nn_dist})
pairs["test_class"] = pairs.test_path.map(cls_of); pairs["train_class"] = pairs.train_path.map(cls_of)
pairs["same_class"] = pairs.test_class == pairs.train_class
pairs["byte_identical"] = (test_df["md5"].values == train_df["md5"].values[nn_idx])
pairs.to_csv(OUT_DIR / "phash_nearest_train_pair_for_every_test_image.csv", index=False)

BINS = [(0, 0), (1, 2), (3, 4), (5, 6), (7, 8), (9, 10), (11, 14)]
rng = np.random.default_rng(0); summ = []
for lo, hi in BINS:
    sel = pairs[(pairs.distance >= lo) & (pairs.distance <= hi)]
    summ.append({"distance bin": f"{lo}-{hi}", "n test images": len(sel),
                 "same class label (%)": round(100 * sel.same_class.mean(), 1) if len(sel) else np.nan,
                 "byte-identical (%)": round(100 * sel.byte_identical.mean(), 1) if len(sel) else np.nan})
    if len(sel) == 0:
        continue
    show = sel.sample(n=min(6, len(sel)), random_state=0)
    fig, axes = plt.subplots(len(show), 2, figsize=(4.4, 2.3 * len(show)))
    axes = np.atleast_2d(axes)
    for r, (_, row) in enumerate(show.iterrows()):
        for c, (p, tag) in enumerate(((row.test_path, "test"), (row.train_path, "nearest train"))):
            axes[r, c].imshow(cv2.imread(p, cv2.IMREAD_GRAYSCALE), cmap="gray"); axes[r, c].axis("off")
            axes[r, c].set_title(f"{tag}: {cls_of(p)}" + (f"\nd={row.distance}" if c == 0 else ""), fontsize=7)
    plt.tight_layout(); plt.savefig(OUT_DIR / f"phash_pairs_d{lo}-{hi}.png", dpi=150); plt.close(fig)
pair_summary = pd.DataFrame(summ); print(pair_summary.to_string(index=False))
pair_summary.to_csv(OUT_DIR / "phash_pair_inspection_summary.csv", index=False)
print("\nHow to read it: two different slices of the same patient are NOT found by pHash at small distances, but re-encoded or "
      "re-cropped copies are. Look at the saved d*.png figures. A bin where almost every pair has the same class label and "
      "looks like the same slice is a true-duplicate bin; a bin where labels agree only about as often as chance is noise.")

## Calibration of the detector
True-copy detection rate for re-encoded / lightly edited copies of training images vs the chance-match rate (nearest neighbour in a different class), per threshold. The recommended threshold is the largest one whose chance-match rate is at most 1 %.

In [ ]:
# ---- Calibrate the pHash detector: how often does it fire on true copies, and how often by chance? ----
# Positive controls: copies made from training images by re-encoding / light edits that keep the same slice.
# Chance-level matches: the nearest neighbour in a DIFFERENT class. Two scans with different labels cannot be the same
# image, so a close cross-class neighbour is a coincidence; its rate at distance t estimates the false-positive rate of a
# threshold t for a pool of this size.
rng = np.random.default_rng(0)
sample_rows = train_df.sample(n=min(300, len(train_df)), random_state=0)

def jpeg(img, q):
    ok, enc = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, q]); return cv2.imdecode(enc, cv2.IMREAD_GRAYSCALE)
def rot(img, a):
    h, w = img.shape; M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0); return cv2.warpAffine(img, M, (w, h))
def shift(img, dx, dy):
    h, w = img.shape; M = np.float32([[1, 0, dx], [0, 1, dy]]); return cv2.warpAffine(img, M, (w, h))
def crop_resize(img, f):
    h, w = img.shape; dh, dw = int(h * (1 - f) / 2), int(w * (1 - f) / 2)
    return cv2.resize(img[dh:h - dh, dw:w - dw], (w, h), interpolation=cv2.INTER_AREA)
TRANSFORMS = {
    "jpeg q=70": lambda im: jpeg(im, 70), "jpeg q=40": lambda im: jpeg(im, 40),
    "resize 0.5 and back": lambda im: cv2.resize(cv2.resize(im, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA), (im.shape[1], im.shape[0])),
    "blur sigma 1": lambda im: cv2.GaussianBlur(im, (0, 0), 1.0),
    "brightness x1.15": lambda im: np.clip(im.astype(np.float32) * 1.15, 0, 255).astype(np.uint8),
    "gaussian noise sd 3": lambda im: np.clip(im.astype(np.float32) + rng.normal(0, 3, im.shape), 0, 255).astype(np.uint8),
    "shift 6 px": lambda im: shift(im, 6, 6), "rotate 3 deg": lambda im: rot(im, 3), "crop 95 % and resize": lambda im: crop_resize(im, 0.95),
}

def phash_of_array(img):
    img = cv2.resize(img, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    low = cv2.dct(img)[:8, :8].flatten(); med = np.median(low[1:])
    return np.uint64(sum(int(b) << i for i, b in enumerate((low > med).astype(np.uint64))))

pos = {k: [] for k in TRANSFORMS}
for _, row in sample_rows.iterrows():
    im = cv2.imread(row["path"], cv2.IMREAD_GRAYSCALE)
    h0 = np.uint64(row["phash"])
    for k, f in TRANSFORMS.items():
        h1 = phash_of_array(f(im))
        pos[k].append(int(POP[np.array([h0 ^ h1], dtype=np.uint64).view(np.uint8)].sum()))
pos = {k: np.array(v) for k, v in pos.items()}

# cross-class nearest-neighbour distance within the training pool
H = train_df["phash"].values.astype(np.uint64); cls = np.array([Path(p).parent.name for p in train_df["path"]])
cross = np.empty(len(H), dtype=np.int16)
for s in range(0, len(H), 256):
    x = (H[s:s + 256, None] ^ H[None, :]).view(np.uint8).reshape(-1, len(H), 8)
    d = POP[x].sum(-1).astype(np.int16)
    d[cls[s:s + 256, None] == cls[None, :]] = 99
    cross[s:s + 256] = d.min(1)

rows = []
for t in range(0, 15):
    r = {"threshold (Hamming <=)": t, "false-positive rate, cross-class neighbour (%)": round(100 * float((cross <= t).mean()), 2),
         "true-copy detection, all edits pooled (%)": round(100 * float(np.mean(np.concatenate(list(pos.values())) <= t)), 1)}
    r.update({f"TPR {k} (%)": round(100 * float((v <= t).mean()), 1) for k, v in pos.items()})
    rows.append(r)
calib = pd.DataFrame(rows); calib.to_csv(OUT_DIR / "phash_calibration.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(calib.iloc[:, :3].to_string(index=False))
ok = calib[calib["false-positive rate, cross-class neighbour (%)"] <= 1.0]
RECOMMENDED_THR = int(ok["threshold (Hamming <=)"].max()) if len(ok) else 0
print(f"\nRecommended threshold: {RECOMMENDED_THR} bits (largest t whose cross-class false-positive rate is <= 1 %). "
      f"Copy it into PHASH_GROUP_THR in notebooks 01-07 and into THR_GROUP below.")
print("Limits: the controls are edits of one image; different slices of the same patient and re-acquisitions are not detected by pHash at any threshold.")
fig, ax = plt.subplots(figsize=(5.4, 3.8))
ax.plot(calib["threshold (Hamming <=)"], calib["true-copy detection, all edits pooled (%)"], marker="o", label="true copies detected")
ax.plot(calib["threshold (Hamming <=)"], calib["false-positive rate, cross-class neighbour (%)"], marker="s", label="cross-class (chance) neighbours")
ax.axvline(RECOMMENDED_THR, color="gray", ls=":"); ax.set_xlabel("pHash threshold (bits)"); ax.set_ylabel("% of images"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "phash_calibration.png", dpi=200); plt.close(fig)

## Groups
Single-linkage clusters at the recommended threshold over train pool + test. The training notebooks use the thumbnail-correlation grouping of the SMGF_Net_7 setup; set `USE_GROUP_SPLIT = True` and `PHASH_GROUP_THR` (pHash helper cell of notebooks 01-07) to this notebook's `RECOMMENDED_THR` to also remove training images that share a pHash cluster with a test image.

In [ ]:
# ---- Group assignment for group-aware splits (same rule as PHASH_GROUP_THR in notebooks 01-07) ----
THR_GROUP = RECOMMENDED_THR if "RECOMMENDED_THR" in globals() else 6     # override by hand if you prefer another value
pool = pd.concat([train_df[["path", "md5", "phash"]].assign(split="train_pool"),
                  test_df[["path", "md5", "phash"]].assign(split="test")], ignore_index=True)
H = pool["phash"].values.astype(np.uint64); n = len(H)
parent = np.arange(n)
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
for s in range(0, n, 256):
    x = (H[s:s + 256, None] ^ H[None, :]).view(np.uint8).reshape(-1, n, 8)
    d = POP[x].sum(-1).astype(np.int16)
    ii, jj = np.nonzero(d <= THR_GROUP); ii = ii + s
    for a, b in zip(ii[ii < jj], jj[ii < jj]):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra
pool["group"] = [find(i) for i in range(n)]
pool["class"] = pool["path"].map(lambda p: Path(p).parent.name)
sizes = pool.groupby("group").size()
span = pool.groupby("group").split.nunique()
cross_groups = span[span > 1].index
leaky_train = pool[(pool.split == "train_pool") & pool.group.isin(cross_groups)]
leaky_test = pool[(pool.split == "test") & pool.group.isin(cross_groups)]
print(f"threshold {THR_GROUP} bits: {sizes.size} groups in {n} images; largest group {sizes.max()}; groups with >1 image: {(sizes > 1).sum()}")
print(f"groups that contain both a training-pool image and a test image: {len(cross_groups)}")
print(f"  training-pool images in those groups: {len(leaky_train)} ({100 * len(leaky_train) / (pool.split == 'train_pool').sum():.2f} %)")
print(f"  test images in those groups:          {len(leaky_test)} ({100 * len(leaky_test) / (pool.split == 'test').sum():.2f} %)")
print("\ntest images in cross-split groups, by class:"); print(leaky_test["class"].value_counts().to_string())
if sizes.max() > 0.05 * n:
    print("\nWARNING: the largest group holds more than 5 % of the pool. Lower THR_GROUP: single linkage is chaining unrelated images.")
pool.drop(columns=["phash"]).to_csv(OUT_DIR / "groups_brisc.csv", index=False)
print("\nSaved groups_brisc.csv. Notebooks 01-07 rebuild the same groups themselves (build_phash_groups), so this file is for inspection "
      "and for the paper's leakage table; nothing needs to be attached to them.")

In [ ]:
import timm, torch
rows = []
for label, name in [("MobileNetV4-Small", "mobilenetv4_conv_small"), ("RepViT-M1", "repvit_m1")]:
    for nc in (1000, 4):
        m = timm.create_model(name, pretrained=False, num_classes=nc)
        rows.append({"model": label, "timm name": name, "classes": nc,
                     "parameters": sum(p.numel() for p in m.parameters()),
                     "parameters (M)": round(sum(p.numel() for p in m.parameters()) / 1e6, 3)})
params = pd.DataFrame(rows); print(params.to_string(index=False))
params.to_csv(OUT_DIR / "baseline_param_counts_4class.csv", index=False)

## For the paper
* Report the row of `phash_internal_summary.csv` at the threshold you choose (a distance of 4 to 6 is a common screen; state the threshold). Say that pHash finds visually similar images, not same-patient images.
* Report `near-duplicate but not byte-identical` for each external set: that is the contamination the MD5 scan missed.
* Use `baseline_param_counts_4class.csv` for the four-class counts of Table 9 (the library 1000-class values there are 3.77M and 4.72M).